# FT-900 半导体出口原报告数值到 Stage1 日历

严格使用同份原报告当月/上月值、报告发行日后的下一个 Qlib 会话；2012-01 定义断点隔离。无市场收益、训练或账户。

In [1]:
import json,hashlib,re
from pathlib import Path
from io import BytesIO
from pypdf import PdfReader
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
root=Path.cwd();card=root/'research/stage1_ft900_export_availability_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['budget']['model_fits']==0
source={}
for key,item in plan['source_objects'].items():
 path=root/item['path'];assert sha(path)==item['sha256'],('source drift',key)
 source[key]=json.loads(path.read_text())
assert source['lead_in_result']['nas_scoped_restore_verified']
assert source['training_readback']['monthly_PDF_hashes_verified']==144
assert source['held_readback']['PDF_hashes_verified']==44
assert source['transition']['status'].startswith('quarantine_January_2012')
lead=source['lead_in_result']['rows'];train=source['training_rows']['rows'];held=source['held_rows']['rows']
assert len(lead)==3 and len(train)==144 and len(held)==44
rows=[]
for x in lead:
 rows.append({'reference_month':x['reference_month'],'sha256':x['source_sha256'],
  'current':x['current_month_million_usd'],'prior':x['prior_month_same_release_million_usd'],
  'release_header':x['release_header']})
for x in train:
 assert x['status']=='unique_numeric_Exhibit7_row' and x['row_candidate_count']==1
 rows.append({'reference_month':x['reference_month'],'sha256':x['source_pdf_sha256'],
  'current':x['current_month_million_usd'],'prior':x['prior_month_same_release_million_usd'],
  'release_header':x['printed_release_header']})
repair=json.loads((card/'mechanical_repair_1.json').read_text())
repair2=json.loads((card/'mechanical_repair_2.json').read_text())
assert sha(card/'plan.json')==repair2['new_plan_sha256']
assert repair2['old_plan_sha256']==repair['new_plan_sha256']
cache_path=card/'held_release_header_cache.json'
assert sha(cache_path)==repair['cache_sha256'] and repair['all_44_headers_parsed']
cache=json.loads(cache_path.read_text())
assert cache['source_row_audit_sha256']==plan['source_objects']['held_rows']['sha256']
headers={x['reference_month']:x for x in cache['rows']}
assert len(headers)==len(held)==44
for x in held:
 assert x['status']=='unique_numeric_Exhibit7_row' and x['row_candidate_count']==1
 assert x['month_identity_matches'] and x['first_page_0830_release_indicated']
 item=headers[x['reference_month']];assert item['source_pdf_sha256']==x['source_pdf_sha256']
 rows.append({'reference_month':x['reference_month'],'sha256':x['source_pdf_sha256'],
  'current':x['current_month_million_usd'],'prior':x['prior_month_same_release_million_usd'],
  'release_header':item['release_header']})
assert len(rows)==191 and len({x['reference_month'] for x in rows})==191
month_order=pd.period_range('2010-10','2026-08',freq='M').astype(str).tolist()
assert [x['reference_month'] for x in rows]==month_order
months='JANUARY FEBRUARY MARCH APRIL MAY JUNE JULY AUGUST SEPTEMBER OCTOBER NOVEMBER DECEMBER'.split()
for x in rows:
 h=x['release_header'];assert '8:30' in h
 m=re.search(r'\b('+'|'.join(months)+r')\s+(\d{1,2}),\s*(\d{4})\b',h,re.I)
 assert m,('release date missing',x['reference_month'],h)
 x['release_date']=pd.Timestamp(f'{m.group(1)} {m.group(2)} {m.group(3)}')
 assert x['prior']>0 and x['current']>=0
 x['mom_pct']=np.nan if x['reference_month']=='2012-01' else 100.0*(x['current']/x['prior']-1.0)
print({'source_events':len(rows),'held_headers_read':len(held),'quarantined_events':sum(pd.isna(x['mom_pct']) for x in rows)})


{'source_events': 191, 'held_headers_read': 44, 'quarantined_events': 1}


In [2]:
qlib.init(provider_uri=str(root/plan['calendar_provider']),region='us',kernels=1,expression_cache=None,dataset_cache=None)
calendar=pd.DatetimeIndex(D.calendar(start_time='2010-12-01',end_time='2026-10-09',freq='day'))
assert calendar.is_monotonic_increasing and calendar.is_unique
for x in rows:
 pos=int(calendar.searchsorted(x['release_date'],side='right'))
 assert pos<len(calendar)
 x['effective_signal_date']=calendar[pos]
assert len({x['effective_signal_date'] for x in rows})==len(rows)
events=pd.DataFrame(rows).sort_values('effective_signal_date').reset_index(drop=True)
train_start,train_end=map(pd.Timestamp,plan['cohorts']['train'])
held_start,held_end=map(pd.Timestamp,plan['cohorts']['held_reused'])
signals=calendar[(calendar>=train_start)&(calendar<=held_end)]
train_mask=(signals>=train_start)&(signals<=train_end)
held_mask=(signals>=held_start)&(signals<=held_end)
assert int(train_mask.sum())==plan['cohorts']['expected_train_sessions']
assert int(held_mask.sum())==plan['cohorts']['expected_held_sessions']
assert len(signals)==int(train_mask.sum()+held_mask.sum())
mapped=pd.merge_asof(pd.DataFrame({'signal_date':signals}),events,
 left_on='signal_date',right_on='effective_signal_date',direction='backward')
mapped['release_age_days']=(mapped.signal_date-mapped.release_date).dt.days
mapped['transition_quarantined']=mapped.reference_month.eq('2012-01')
assert mapped.reference_month.notna().all()
assert mapped.mom_pct.isna().equals(mapped.transition_quarantined)
assert mapped.release_age_days.ge(0).all()
assert mapped.signal_date.is_unique and mapped.signal_date.is_monotonic_increasing
static=pd.concat({'feature':mapped.set_index('signal_date')[['mom_pct']].rename(columns={'mom_pct':plan['feature_name']}),
                  'meta':mapped.set_index('signal_date')[['release_age_days','transition_quarantined']]},axis=1)
static.index=pd.MultiIndex.from_arrays([mapped.signal_date,['SOXL_SOXX_CASH']*len(mapped)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['cohorts']['train']),'held':tuple(plan['cohorts']['held_reused'])})
tr=dataset.prepare('train',col_set=['feature','meta']);he=dataset.prepare('held',col_set=['feature','meta'])
assert len(tr)==int(train_mask.sum()) and len(he)==int(held_mask.sum())
audit={'status':'conditional_FT900_source_calendar_alignment_complete_no_fit','plan_sha256':sha(card/'plan.json'),
 'source_events':len(events),'event_reference_bounds':[events.reference_month.min(),events.reference_month.max()],
 'train_signal_sessions':len(tr),'held_signal_sessions':len(he),
 'train_nonmissing_feature_sessions':int(mapped.loc[train_mask,'mom_pct'].notna().sum()),
 'held_nonmissing_feature_sessions':int(mapped.loc[held_mask,'mom_pct'].notna().sum()),
 'transition_quarantine_sessions':int(mapped.transition_quarantined.sum()),
 'transition_quarantine_first':str(mapped.loc[mapped.transition_quarantined,'signal_date'].min().date()),
 'transition_quarantine_last':str(mapped.loc[mapped.transition_quarantined,'signal_date'].max().date()),
 'max_source_age_days':int(mapped.release_age_days.max()),
 'first_signal_source':mapped.iloc[0].reference_month,'last_signal_source':mapped.iloc[-1].reference_month,
 'historical_first_public_bytes_qualified':False,'market_outcome_reads':0,'factor_fits':0,'native_accounts':0}
(card/'availability_audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n')
print({k:audit[k] for k in ('source_events','train_signal_sessions','held_signal_sessions','train_nonmissing_feature_sessions','held_nonmissing_feature_sessions','transition_quarantine_sessions','first_signal_source','last_signal_source')})


[38720:MainThread](2026-10-23 00:45:50,465) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[38720:MainThread](2026-10-23 00:45:50,467) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[38720:MainThread](2026-10-23 00:45:50,467) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


[38720:MainThread](2026-10-23 00:45:50,480) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[38720:MainThread](2026-10-23 00:45:50,481) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[38720:MainThread](2026-10-23 00:45:50,481) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


{'source_events': 191, 'train_signal_sessions': 3020, 'held_signal_sessions': 944, 'train_nonmissing_feature_sessions': 2997, 'held_nonmissing_feature_sessions': 944, 'transition_quarantine_sessions': 23, 'first_signal_source': '2010-10', 'last_signal_source': '2026-08'}


In [3]:
import pickle,tarfile,tempfile,subprocess,shlex
from src.data.nas_market_data import archive_nas_recorder,_settings
tracking=card/'qlib_recorder';summary_path=card/'summary.json'
dataset.config(dump_all=True,recursive=True)
roundtrip=pickle.loads(pickle.dumps(dataset))
assert len(roundtrip.prepare('train',col_set=['feature','meta']))==3020
assert len(roundtrip.prepare('held',col_set=['feature','meta']))==944
if summary_path.exists():
 old_summary=json.loads(summary_path.read_text())
 failure=json.loads((card/'readback_attempt1_failure.json').read_text())
 assert old_summary['recorder_id']==failure['recorder_id'] and old_summary['experiment_id']==failure['experiment_id']
 assert failure['old_plan_sha256']==repair2['old_plan_sha256']
 state_path=card/'repair_run_state.json'
 assert not state_path.exists(), 'fixed Recorder resume already attempted; inspect state before any retry'
 state={'status':'resume_attempted','recorder_id':old_summary['recorder_id'],'experiment_id':old_summary['experiment_id'],
        'repair_plan_sha256':repair2['new_plan_sha256'],'model_fits':0,'native_accounts':0}
 state_path.write_text(json.dumps(state,ensure_ascii=False,indent=2)+'\n')
 with R.uri_context(str(tracking)):
  original_rec=R.get_recorder(recorder_id=state['recorder_id'],experiment_id=state['experiment_id'])
  oldmapped=original_rec.load_object('analysis/mapped_signals')
 assert oldmapped.reference_month.equals(mapped.reference_month)
 assert np.allclose(oldmapped.mom_pct.to_numpy(float),mapped.mom_pct.to_numpy(float),equal_nan=True)
 with R.start(experiment_name='stage1_ft900_export_availability_20261023',uri=str(tracking),
              recorder_id=state['recorder_id'],resume=True):
  rec=R.get_recorder()
  assert rec.id==state['recorder_id'] and str(rec.experiment_id)==state['experiment_id']
  rec.save_objects(artifact_path='metadata',dataset=dataset)
  rec.log_metrics(dataset_repair=1,model_fits=0,native_accounts=0)
 recorder_id=state['recorder_id'];experiment_id=state['experiment_id'];summary=old_summary
else:
 with R.start(experiment_name='stage1_ft900_export_availability_20261023',uri=str(tracking)):
  rec=R.get_recorder()
  rec.save_objects(artifact_path='inputs',plan=plan,source_identity={k:v['sha256'] for k,v in plan['source_objects'].items()})
  rec.save_objects(artifact_path='analysis',events=events,mapped_signals=mapped,availability_audit=audit)
  rec.save_objects(artifact_path='metadata',dataset=dataset)
  rec.log_metrics(source_events=len(events),train_sessions=len(tr),held_sessions=len(he),model_fits=0,native_accounts=0)
  recorder_id=rec.id;experiment_id=str(rec.experiment_id)
 summary={'experiment':'stage1_ft900_export_availability_20261023','decision':'conditional_source_calendar_alignment_complete_information_and_PIT_unqualified',
  'objective_achieved':False,'result':'availability_audit.json','source_events':len(events),
  'train_signal_sessions':len(tr),'held_signal_sessions':len(he),'transition_quarantine_sessions':audit['transition_quarantine_sessions'],
  'historical_first_public_bytes_qualified':False,'label_maturity_checked':False,'model_fits':0,'native_accounts':0,
  'recorder_id':recorder_id,'experiment_id':experiment_id,'recorder_uri':str(tracking),
  'recorder_local_dir':str(tracking/experiment_id/recorder_id),'artifact_policy':plan['artifact_policy']}
with R.uri_context(str(tracking)):
 checked=R.get_recorder(recorder_id=recorder_id,experiment_id=experiment_id)
 saved=checked.load_object('metadata/dataset')
 assert len(saved.prepare('train',col_set=['feature','meta']))==3020
 assert len(saved.prepare('held',col_set=['feature','meta']))==944
 assert checked.load_object('analysis/mapped_signals').reference_month.equals(mapped.reference_month)
artifact=tracking/experiment_id/recorder_id/'artifacts/metadata/dataset'
new_dataset_sha=sha(artifact)
if summary_path.exists():
 assert new_dataset_sha!=failure['old_dataset_artifact_sha256']
state_path=card/'repair_run_state.json'
if state_path.exists():
 state=json.loads(state_path.read_text());state['status']='dataset_readback_passed';state['new_dataset_artifact_sha256']=new_dataset_sha
 state_path.write_text(json.dumps(state,ensure_ascii=False,indent=2)+'\n')
settings=_settings(root)
binding_path=card/'nas_binding.json'
if not binding_path.exists():
 binding={'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':'stage1_ft900_export_availability_20261023','ssh':settings['ssh'],'source_plan':'plan.json'}
 binding_path.write_text(json.dumps(binding,ensure_ascii=False,indent=2)+'\n')
evidence=archive_nas_recorder(binding_path,workspace_root=root,card=card,recorder_id=recorder_id,experiment_id=experiment_id)
remote=evidence['nas_path'];digest=evidence['sha256']
with tempfile.TemporaryDirectory(prefix='ft900-map-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:
  run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==digest
 with tarfile.open(archive,'r:gz') as source_tar:
  names=[m.name for m in source_tar.getmembers() if m.isfile()]
 assert any(name.endswith('/artifacts/metadata/dataset') for name in names)
summary['nas_evidence']=evidence
summary['nas_readback']={'sha256':digest,'bytes':evidence['size_bytes'],'dataset_artifact_present':True,'verified':True}
summary['dataset_artifact_sha256']=new_dataset_sha
summary['mechanical_repair_2']='mechanical_repair_2.json'
summary['plan_repair_sha256']=repair2['new_plan_sha256']
summary_path.write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
report={'status':'native_Qlib_DatasetH_readback_and_NAS_restore_passed','recorder_id':recorder_id,'experiment_id':experiment_id,
 'mapped_signals':len(mapped),'train_DatasetH_rows':3020,'held_DatasetH_rows':944,'quarantined_rows':audit['transition_quarantine_sessions'],
 'dataset_artifact_sha256':new_dataset_sha,'nas_archive_sha256':digest,'nas_remote_readback_verified':True,
 'model_fits':0,'native_accounts':0}
(card/'readback_verification.json').write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n')
if state_path.exists():
 state=json.loads(state_path.read_text());state['status']='completed';state['new_nas_archive_sha256']=digest
 state_path.write_text(json.dumps(state,ensure_ascii=False,indent=2)+'\n')
print({'recorder_id':recorder_id,'dataset_readback':True,'new_nas_sha256':digest})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[38720:MainThread](2026-10-23 00:45:51,884) INFO - qlib.workflow - [exp.py:258] - Experiment 993438492998129634 starts running ...


[38720:MainThread](2026-10-23 00:45:51,890) INFO - qlib.workflow - [recorder.py:345] - Recorder e96807ef987d4abcb048dd350abc9428 starts running under Experiment 993438492998129634 ...


Exception in thread Thread-5 (run):
Traceback (most recent call last):
  File "/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/client.py", line 413, in log_param
    self.store.log_param(run_id, param)
  File "/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/store/tracking/file_store.py", line 1130, in log_param
    self._log_run_param(run_info, param)
  File "/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/store/tracking/file_store.py", line 1136, in _log_run_param
    self._validate_new_param_value(
  File "/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/store/tracking/file_store.py", line 1156, in _validate_new_param_value
    raise MlflowException(
mlflow.exceptions.MlflowException: Changing param values is not allowed. Param with key='cmd-sys.argv' was already logged with value='/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/ipy

[38720:MainThread](2026-10-23 00:45:53,124) INFO - qlib.timer - [log.py:127] - Time cost: 0.003s | waiting `async_log` Done


{'recorder_id': 'e96807ef987d4abcb048dd350abc9428', 'dataset_readback': True, 'new_nas_sha256': '11ff39d62ee840c308ea83311f8cd9448b498b7f63850d206e331491aa647b03'}


## 中文结论

原始 FT-900 同报告数值共 191 次发行事件，以发行日期后的下个 Qlib 会话对齐原 Stage1 的 3020 条训练和 944 条复用诊断信号。训练中 23 条因 2012-01 重分类留空，诊断 944/944 有值；来源最大日龄 76 天。最终 Qlib Recorder `e96807ef987d4abcb048dd350abc9428` 原生读回 train/held 3020/944 行，NAS SHA‑256 `11ff39d62ee840c308ea83311f8cd9448b498b7f63850d206e331491aa647b03` 已读回。

两次实现问题及预算已保留：报告日期逗号空格解析、DatasetH 持久化；均未消耗模型或账户。结果仅为来源可用性，不证明历史首次公开、预测增量或 SOXX 超额。风险、决定和下一步见 `conclusion.md`、`summary.json` 与 `readback_verification.json`。
